# 💬 Intern Feedback Sentiment Analysis
### Analyzing intern feedback to identify positive and negative sentiments
**Model:** Logistic Regression + Visualization  
**Dataset:** 300 Intern Feedback Records  
**Target:** Sentiment → Positive / Negative / Neutral

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## Step 2 — Load Dataset

In [ ]:
df = pd.read_csv('intern_feedback_dataset.csv')
print('Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

## Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
print('=== Null Values ===')
print(df.isnull().sum())
print('\n=== Sentiment Distribution ===')
print(df['sentiment'].value_counts())

In [ ]:
# Sentiment distribution pie chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
colors = ['#2ecc71', '#e74c3c', '#f39c12']
df['sentiment'].value_counts().plot(
    kind='pie', ax=axes[0], autopct='%1.1f%%',
    colors=colors, startangle=140
)
axes[0].set_title('Sentiment Distribution')
axes[0].set_ylabel('')

# Bar chart by department
dept_sent = df.groupby(['department', 'sentiment']).size().unstack(fill_value=0)
dept_sent.plot(kind='bar', ax=axes[1], color=colors, edgecolor='black')
axes[1].set_title('Sentiment by Department')
axes[1].set_xlabel('Department')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Sentiment')

plt.tight_layout()
plt.show()

In [ ]:
# Rating distribution by sentiment
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x='sentiment', y='rating',
            palette={'Positive': '#2ecc71', 'Negative': '#e74c3c', 'Neutral': '#f39c12'})
plt.title('Rating Distribution by Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Rating (1-5)')
plt.tight_layout()
plt.show()

In [ ]:
# Would recommend analysis
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Recommend by sentiment
rec = df.groupby('sentiment')['would_recommend'].mean() * 100
rec.plot(kind='bar', ax=axes[0],
         color=['#e74c3c', '#f39c12', '#2ecc71'], edgecolor='black')
axes[0].set_title('% Who Would Recommend by Sentiment')
axes[0].set_ylabel('Percentage %')
axes[0].tick_params(axis='x', rotation=0)

# Work life balance vs sentiment
sns.boxplot(data=df, x='sentiment', y='work_life_balance', ax=axes[1],
            palette={'Positive': '#2ecc71', 'Negative': '#e74c3c', 'Neutral': '#f39c12'})
axes[1].set_title('Work Life Balance Score by Sentiment')

plt.tight_layout()
plt.show()

## Step 4 — Text Preprocessing & Word Analysis

In [ ]:
# Most common words in positive feedback
from sklearn.feature_extraction.text import CountVectorizer

def get_top_words(sentiment, n=15):
    texts = df[df['sentiment'] == sentiment]['feedback_text'].tolist()
    cv = CountVectorizer(stop_words='english', max_features=n)
    cv.fit_transform(texts)
    return list(cv.vocabulary_.keys())

pos_words = get_top_words('Positive')
neg_words = get_top_words('Negative')

print('Top words in POSITIVE feedback:')
print(pos_words)
print('\nTop words in NEGATIVE feedback:')
print(neg_words)

## Step 5 — Build Sentiment Classification Model

In [ ]:
# Prepare data
x = df['feedback_text']
y = df['sentiment']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

print('Train size:', x_train.shape)
print('Test size :', x_test.shape)
print('\nTrain label distribution:')
print(y_train.value_counts())

In [ ]:
# Build Pipeline: TF-IDF + Logistic Regression
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=500,
        ngram_range=(1, 2),
        stop_words='english'
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        random_state=42,
        C=1.0
    ))
])

pipeline.fit(x_train, y_train)
print('Model trained successfully!')

## Step 6 — Model Evaluation

In [ ]:
y_pred = pipeline.predict(x_test)

print('=' * 50)
print('         CLASSIFICATION RESULTS')
print('=' * 50)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print('=' * 50)
print('\nDetailed Report:')
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix
plt.figure(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred, labels=['Positive', 'Negative', 'Neutral'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Positive', 'Negative', 'Neutral'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix - Sentiment Classification')
plt.tight_layout()
plt.show()

In [ ]:
# Cross Validation
cv_scores = cross_val_score(pipeline, x, y, cv=5, scoring='accuracy')
print(f'Cross-Val Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
print(f'All fold scores: {[round(s, 4) for s in cv_scores]}')

## Step 7 — Identify Improvement Areas

In [ ]:
# Areas where interns are most dissatisfied
neg_df = df[df['sentiment'] == 'Negative']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Negative feedback by department
neg_by_dept = neg_df['department'].value_counts()
neg_by_dept.plot(kind='bar', ax=axes[0], color='#e74c3c', edgecolor='black')
axes[0].set_title('Negative Feedback Count by Department')
axes[0].set_xlabel('Department')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Mentor availability vs sentiment
mentor_sent = df.groupby(['mentor_available', 'sentiment']).size().unstack(fill_value=0)
mentor_sent.plot(kind='bar', ax=axes[1],
                  color=['#e74c3c', '#f39c12', '#2ecc71'], edgecolor='black')
axes[1].set_title('Mentor Availability vs Sentiment')
axes[1].set_xlabel('Mentor Available')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Sentiment')

plt.tight_layout()
plt.show()

In [ ]:
# Improvement areas summary
print('=' * 55)
print('        INTERN SATISFACTION IMPROVEMENT AREAS')
print('=' * 55)

print(f"\nOverall Positive Rate : {(df['sentiment']=='Positive').mean()*100:.1f}%")
print(f"Overall Negative Rate : {(df['sentiment']=='Negative').mean()*100:.1f}%")
print(f"Overall Neutral Rate  : {(df['sentiment']=='Neutral').mean()*100:.1f}%")

print("\n--- Department needing most improvement ---")
print(neg_by_dept.head(3).to_string())

print("\n--- Avg Rating by Sentiment ---")
print(df.groupby('sentiment')['rating'].mean().round(2).to_string())

print("\n--- Work Life Balance by Sentiment ---")
print(df.groupby('sentiment')['work_life_balance'].mean().round(2).to_string())

print("\n--- Learning Score by Sentiment ---")
print(df.groupby('sentiment')['learning_score'].mean().round(2).to_string())

## Step 8 — Predict on New Feedback

In [ ]:
# Test with new feedback
new_feedbacks = [
    "The internship was fantastic, I learned so much and the team was great",
    "Very poor experience, no guidance and management was rude",
    "It was an average experience, some things were good some were not",
    "I loved the projects and my mentor was always available to help me",
    "The work was boring and repetitive with no real learning opportunities"
]

predictions = pipeline.predict(new_feedbacks)
probabilities = pipeline.predict_proba(new_feedbacks)

print('=== NEW FEEDBACK PREDICTIONS ===')
for i, (fb, pred, prob) in enumerate(zip(new_feedbacks, predictions, probabilities)):
    print(f'\nFeedback {i+1}: "{fb[:60]}..."')
    print(f'Prediction  : {pred}')
    classes = pipeline.classes_
    for cls, p in zip(classes, prob):
        print(f'  {cls}: {p*100:.1f}%')

In [ ]:
# Final summary chart
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Overall sentiment
sent_counts = df['sentiment'].value_counts()
axes[0].pie(sent_counts, labels=sent_counts.index, autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c', '#f39c12'])
axes[0].set_title('Overall Sentiment')

# Avg rating per dept
dept_rating = df.groupby('department')['rating'].mean().sort_values()
dept_rating.plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Avg Rating by Department')
axes[1].set_xlabel('Average Rating')

# Would recommend
rec_counts = df['would_recommend'].map({1: 'Yes', 0: 'No'}).value_counts()
rec_counts.plot(kind='bar', ax=axes[2],
                color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[2].set_title('Would Recommend Internship?')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=0)

plt.suptitle('Intern Feedback Analysis - Final Summary', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n Project Complete!')